# YOLOv8s Multi-Class Detection

**Classes**: Airplane, Drone, Helicopter  
**Optimized for**: Mac M4 with 24GB RAM  
**Training Time**: ~3-5 hours (5k images)

## Setup

In [6]:
%matplotlib inline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✓ Setup complete')

✓ Setup complete


## Install/Update Ultralytics

In [7]:
!pip install -U ultralytics


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Load YOLOv8s Model

In [8]:
from ultralytics import YOLO

# Resume from checkpoint (training3 completed 1 epoch)
checkpoint_path = 'runs/detect/yolov8s_multiclass_results/training3/weights/last.pt'
model = YOLO(checkpoint_path)
print(f'✓ Loaded checkpoint from {checkpoint_path}')
print('  Training will resume from where it left off')

✓ Loaded checkpoint from runs/detect/yolov8s_multiclass_results/training3/weights/last.pt
  Training will resume from where it left off


## Train (Mac M4 Optimized)

In [10]:
# Training configuration - Mac M4 24GB RAM optimized
results = model.train(
    data='/Users/hilkin/.cache/kagglehub/datasets/cybersimar08/drone-detection/versions/3/drone-detection-new.v5-new-train.yolov8/data.yaml',
    epochs=30,
    imgsz=640,
    batch=24,         # Optimized for 24GB RAM (increased from 16)
    device='mps',     # Mac M4 GPU
    workers=10,       # 10 cores
    cache='ram',      # Cache in RAM
    patience=25,
    save_period=10,
    amp=True,
    plots=True,
    project='yolov8s_multiclass_results',
    name='training_resumed'  # ← CHANGED THIS LINE ONLY
)

print('\n✓ Training complete!')

Ultralytics 8.4.14 🚀 Python-3.14.2 torch-2.10.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/hilkin/.cache/kagglehub/datasets/cybersimar08/drone-detection/versions/3/drone-detection-new.v5-new-train.yolov8/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/yolov8s_multiclass_results/training3/weights/last.pt, momentum=0.937, mosaic=1.0, multi

KeyboardInterrupt: 

## Evaluate

In [ ]:
metrics = model.val()

print(f'\n📊 Overall Performance:')
print(f'  mAP50:     {metrics.box.map50:.4f}')
print(f'  mAP50-95:  {metrics.box.map:.4f}')
print(f'  Precision: {metrics.box.mp:.4f}')
print(f'  Recall:    {metrics.box.mr:.4f}')

# Per-class metrics
print(f'\n📊 Per-Class mAP50:')
for i, name in enumerate(['Airplane', 'Drone', 'Helicopter']):
    print(f'  {name}: {metrics.box.maps[i]:.4f}')

## Test Predictions

In [ ]:
import glob
import cv2
import random

val_dir = '/Users/hilkin/.cache/kagglehub/datasets/cybersimar08/drone-detection/versions/3/drone-detection-new.v5-new-train.yolov8/valid/images'
all_imgs = glob.glob(f'{val_dir}/*')
test_imgs = random.sample(all_imgs, min(6, len(all_imgs)))

results = model.predict(test_imgs, conf=0.25, device='mps', verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

class_names = ['Airplane', 'Drone', 'Helicopter']

for idx, result in enumerate(results):
    img_bgr = result.plot()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[idx].imshow(img_rgb)
    axes[idx].axis('off')
    
    # Count detections per class
    detections = {}
    for box in result.boxes:
        cls_id = int(box.cls[0])
        cls_name = class_names[cls_id]
        detections[cls_name] = detections.get(cls_name, 0) + 1
    
    title = ', '.join([f'{k}:{v}' for k,v in detections.items()]) if detections else 'No detections'
    axes[idx].set_title(title, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()
print('✓ Predictions displayed')

## Training Results

In [ ]:
from IPython.display import Image, display
import os

results_img = 'yolov8s_multiclass_results/training/results.png'
if os.path.exists(results_img):
    display(Image(filename=results_img, width=1200))
else:
    print('Results image not found')

## Export Model

In [ ]:
# Export to different formats
model.export(format='onnx')
model.export(format='coreml')
print('✓ Model exported')

## Summary

**Model**: `yolov8s_multiclass_results/training/weights/best.pt`  
**Classes**: Airplane, Drone, Helicopter  
**Optimized for**: Mac M4, 24GB RAM